In [1]:
# compute coreference metrics

# for each doc: tanh (avg chain size / num chains)
# averaged across docs

In [2]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

In [3]:
MODELS = ['human', 'claude45', 'gpt4o', 'internvl3', 'llama4scout', 'qwen3vl']
PROMPTS = ['original', 'large']
COREF_DIR = Path('/mimer/NOBACKUP/groups/naiss2025-22-1187/coherence-driven-humans/models/linkappend/data-out/conll_to_json')

# story_id=5540, seed=42, P(AI)=0.9995
# story_id=10499, seed=42, P(AI)=0.9991
# story_id=7195, seed=43, P(AI)=0.9988
# story_id=3761, seed=42, P(AI)=0.9949
# story_id=1214, seed=42, P(AI)=0.9936
# story_id=5047, seed=42, P(AI)=0.9539
# story_id=3891, seed=43, P(AI)=0.7180

ROBERTA_EXCLUDED_STORY_IDS = [1214, 3761, 5047, 5540, 7195, 10499]

In [4]:
def load_jsonlines(filepath):
    documents = []
    with open(filepath, 'r') as f:
        for line in f:
            documents.append(json.loads(line))
    return documents

def clean_human_original_data(doc):
    # human original data has [SENT] in the end, need to remove it, an artefact from some processing
    if len(doc['sentences']) > 0:
        last_sent = doc['sentences'][-1]
        if len(last_sent) >= 3 and last_sent[-3:] == ["[", "SENT", "]"]:
            doc['sentences'][-1] = last_sent[:-3]
            num_tokens_before_last = sum(len(sent) for sent in doc['sentences'][:-1])
            removed_token_start = num_tokens_before_last + len(last_sent) - 3
            new_clusters = []
            for cluster in doc['clusters']:
                new_mentions = [m for m in cluster if m[0] < removed_token_start]
                if new_mentions:
                    new_clusters.append(new_mentions)
            doc['clusters'] = new_clusters
    return doc

def normalize_story_id(raw_id):
    # formatting of story id
    s = str(raw_id).replace('story_', '')
    parts = s.split('_')
    if len(parts) >= 2 and parts[0] == 'doc':
        s = parts[1]
    try:
        return int(float(s))
    except (ValueError, TypeError):
        return None

def compute_coref_metrics(doc):
    clusters = doc['clusters']
    num_chains = len(clusters)
    chain_sizes = [len(cluster) for cluster in clusters]
    avg_chain_size = np.mean(chain_sizes) if chain_sizes else 0
    epsilon = 0.01
    raw_ratio = avg_chain_size / (num_chains + epsilon) if num_chains > 0 else 0
    coref_ratio = np.tanh(raw_ratio)
    return {
        'num_chains': num_chains,
        'avg_chain_size': avg_chain_size,
        'coref_ratio': coref_ratio
    }

In [5]:
def load_coref_data():
    all_data = []
    for model in MODELS:
        for prompt in PROMPTS:
            if model == 'human' and prompt == 'large':
                seeds_to_load = ['42', '43', '44']
            else:
                seeds_to_load = ['42']
            for seed in seeds_to_load:
                filepath = COREF_DIR / f"{model}_{prompt}_seed{seed}_test_snapshots__local_json-nopound_pred.jsonlines"
                if not filepath.exists():
                    print(f"  Warning: {filepath.name} not found")
                    continue
                documents = load_jsonlines(filepath)
                for doc in documents:
                    if model == 'human' and prompt == 'original':
                        doc = clean_human_original_data(doc)
                    metrics = compute_coref_metrics(doc)
                    story_id = normalize_story_id(doc['doc_key'])
                    all_data.append({
                        'model': model, 'prompt': prompt, 'seed': seed,
                        'story_id': story_id, **metrics
                    })
    df = pd.DataFrame(all_data)
    df['story_id'] = df['story_id'].astype(int)
    return df

def prepare_coref_data(df, excluded_story_ids=None):
    results = []
    for model in MODELS:
        for prompt in PROMPTS:
            df_mp = df[(df['model'] == model) & (df['prompt'] == prompt)]
            if len(df_mp) == 0:
                continue
            if model == 'human' and prompt == 'large':
                df_avg = df_mp.groupby('story_id')[['num_chains', 'avg_chain_size', 'coref_ratio']].mean().reset_index()
                for _, row in df_avg.iterrows():
                    results.append({'model': model, 'prompt': prompt, 'story_id': row['story_id'],
                                    'num_chains': row['num_chains'], 'avg_chain_size': row['avg_chain_size'],
                                    'coref_ratio': row['coref_ratio']})
            else:
                df_seed42 = df_mp[df_mp['seed'] == '42']
                for _, row in df_seed42.iterrows():
                    results.append({'model': model, 'prompt': prompt, 'story_id': row['story_id'],
                                    'num_chains': row['num_chains'], 'avg_chain_size': row['avg_chain_size'],
                                    'coref_ratio': row['coref_ratio']})

    result = pd.DataFrame(results)
    result['story_id'] = result['story_id'].astype(int)

    if excluded_story_ids:
        excluded_story_ids = {int(x) for x in excluded_story_ids}
        result = result[~result['story_id'].isin(excluded_story_ids)].copy()

    return result

In [6]:
df_coref_raw = load_coref_data()
print(df_coref_raw.head())

   model    prompt seed  story_id  num_chains  avg_chain_size  coref_ratio
0  human  original   42     13683           2            7.50     0.998852
1  human  original   42     13596           5            3.60     0.616018
2  human  original   42     12423           4            4.75     0.828877
3  human  original   42     12358           2            6.50     0.996899
4  human  original   42     11719           2            6.00     0.994905


In [7]:
df_coref = prepare_coref_data(df_coref_raw)
df_coref_54 = prepare_coref_data(df_coref_raw, excluded_story_ids=ROBERTA_EXCLUDED_STORY_IDS)

print(df_coref.head())
print(df_coref.shape, df_coref['story_id'].nunique())

print(df_coref_54.head())
print(df_coref_54.shape, df_coref_54['story_id'].nunique())


   model    prompt  story_id  num_chains  avg_chain_size  coref_ratio
0  human  original     13683         2.0            7.50     0.998852
1  human  original     13596         5.0            3.60     0.616018
2  human  original     12423         4.0            4.75     0.828877
3  human  original     12358         2.0            6.50     0.996899
4  human  original     11719         2.0            6.00     0.994905
(720, 6) 60
   model    prompt  story_id  num_chains  avg_chain_size  coref_ratio
0  human  original     13683         2.0            7.50     0.998852
1  human  original     13596         5.0            3.60     0.616018
2  human  original     12423         4.0            4.75     0.828877
3  human  original     12358         2.0            6.50     0.996899
4  human  original     11719         2.0            6.00     0.994905
(648, 6) 54


In [8]:
# coreference aggregate metrics on original prompt, all 60
df_original_60 = df_coref[df_coref['prompt'] == 'original'].copy()

coref_agg_original_60 = (
    df_original_60.groupby('model')
    .agg(
        num_chains=('num_chains', 'mean'),
        avg_chain_size=('avg_chain_size', 'mean'),
        coref_ratio_mean=('coref_ratio', 'mean'),
        coref_ratio_std=('coref_ratio', 'std'),
        count=('coref_ratio', 'count')
    )
    .reset_index()
    .sort_values('model')
)

# coreference aggregate metrics on original prompt, 54
df_original_54 = df_coref_54[df_coref_54['prompt'] == 'original'].copy()

coref_agg_original_54 = (
    df_original_54.groupby('model')
    .agg(
        num_chains=('num_chains', 'mean'),
        avg_chain_size=('avg_chain_size', 'mean'),
        coref_ratio_mean=('coref_ratio', 'mean'),
        coref_ratio_std=('coref_ratio', 'std'),
        count=('coref_ratio', 'count')
    )
    .reset_index()
    .sort_values('model')
)

# coreference aggregate metrics on large prompt, 54
df_large_54 = df_coref_54[df_coref_54['prompt'] == 'large'].copy()

coref_agg_large_54 = (
    df_large_54.groupby('model')
    .agg(
        num_chains=('num_chains', 'mean'),
        avg_chain_size=('avg_chain_size', 'mean'),
        coref_ratio_mean=('coref_ratio', 'mean'),
        coref_ratio_std=('coref_ratio', 'std'),
        count=('coref_ratio', 'count')
    )
    .reset_index()
    .sort_values('model')
)

In [9]:
coref_agg_original_60

,model,num_chains,avg_chain_size,coref_ratio_mean,coref_ratio_std,count
0,claude45,6.900000,4.607905,0.582792,0.181004,60
1,gpt4o,5.883333,4.729590,0.647946,0.210240,60
2,human,3.800000,4.666290,0.773725,0.186228,60
3,internvl3,7.050000,5.263807,0.631249,0.182304,60
4,llama4scout,4.866667,5.428265,0.730055,0.233890,60
5,qwen3vl,4.583333,4.333115,0.703818,0.203740,60


In [ ]:
coref_agg_original_54

,model,num_chains,avg_chain_size,coref_ratio_mean,coref_ratio_std,count
0,claude45,6.740741,4.664060,0.595288,0.181967,54
1,gpt4o,6.000000,4.740329,0.642351,0.207701,54
2,human,3.740741,4.792174,0.785605,0.187497,54
3,internvl3,7.018519,5.237254,0.630168,0.181711,54
4,llama4scout,4.796296,5.335053,0.732437,0.230584,54
5,qwen3vl,4.648148,4.357165,0.699600,0.199781,54


In [11]:
coref_agg_large_54

,model,num_chains,avg_chain_size,coref_ratio_mean,coref_ratio_std,count
0,claude45,7.425926,5.010426,0.586767,0.194715,54
1,gpt4o,5.500000,4.421223,0.660675,0.194099,54
2,human,6.123457,5.366601,0.684722,0.139324,54
3,internvl3,7.314815,4.593478,0.561352,0.194395,54
4,llama4scout,5.962963,4.394267,0.630909,0.210409,54
5,qwen3vl,5.851852,4.271700,0.619563,0.173542,54


In [12]:
from scipy.stats import ttest_rel

# paired t-test between humans and models
def human_model_paired_ttests(df_subset):
    rows = []
    df_human = df_subset[df_subset['model'] == 'human'].set_index('story_id')

    for model in MODELS:
        if model == 'human':
            continue

        df_model = df_subset[df_subset['model'] == model].set_index('story_id')
        common_ids = sorted(df_human.index.intersection(df_model.index))

        if len(common_ids) < 2:
            rows.append({
                'model': model,
                't_stat': np.nan,
                'p_value': np.nan,
                'n': len(common_ids)
            })
            continue

        human_vals = df_human.loc[common_ids, 'coref_ratio'].values
        model_vals = df_model.loc[common_ids, 'coref_ratio'].values
        t_stat, p_value = ttest_rel(human_vals, model_vals)

        rows.append({
            'model': model,
            't_stat': float(t_stat),
            'p_value': float(p_value),
            'n': len(common_ids)
        })

    result = pd.DataFrame(rows).sort_values('model').reset_index(drop=True)
    result['t_stat'] = result['t_stat'].map(lambda x: f"{x:.2f}" if pd.notna(x) else np.nan)
    result['p_value'] = result['p_value'].map(lambda x: f"{x:.3f}" if pd.notna(x) else np.nan)
    return result.set_index('model')[['t_stat', 'p_value', 'n']]

ttest_original_60 = human_model_paired_ttests(df_original_60)
display(ttest_original_60)

,t_stat,p_value,n
model,,,
claude45,6.90,0.000,60
gpt4o,4.60,0.000,60
internvl3,4.60,0.000,60
llama4scout,1.12,0.268,60
qwen3vl,2.22,0.030,60


In [13]:
ttest_original_54 = human_model_paired_ttests(df_original_54)
display(ttest_original_54)

ttest_large_54 = human_model_paired_ttests(df_large_54)
display(ttest_large_54)

,t_stat,p_value,n
model,,,
claude45,6.23,0.000,54
gpt4o,5.14,0.000,54
internvl3,4.63,0.000,54
llama4scout,1.31,0.196,54
qwen3vl,2.67,0.010,54


,t_stat,p_value,n
model,,,
claude45,3.93,0.000,54
gpt4o,0.97,0.334,54
internvl3,5.48,0.000,54
llama4scout,1.67,0.101,54
qwen3vl,3.20,0.002,54


In [14]:
corefdata_path = Path('./analysis_data/coreference/')
corefdata_path.mkdir(parents=True, exist_ok=True)

df_coref_raw.to_csv(corefdata_path / 'coref_metrics_raw.csv', index=False)
df_coref.to_csv(corefdata_path / 'coref_metrics.csv', index=False)
coref_agg_original_60.to_csv(corefdata_path / 'coref_metrics_agg_original_60.csv', index=False)

df_coref_54.to_csv(corefdata_path / 'coref_metrics_54.csv', index=False)
coref_agg_original_54.to_csv(corefdata_path / 'coref_metrics_agg_original_54.csv', index=False)
coref_agg_large_54.to_csv(corefdata_path / 'coref_metrics_agg_large_54.csv', index=False)

ttest_original_60.to_csv(corefdata_path / 'coref_ttest_original_60.csv', index=False)
ttest_original_54.to_csv(corefdata_path / 'coref_ttest_original_54.csv', index=False)
ttest_large_54.to_csv(corefdata_path / 'coref_ttest_large_54.csv', index=False)